# FER2013 - Advanced Approach with Transfer Learning

**Different Approach**: Using Transfer Learning + Data Augmentation

**Key Differences from Basic Approach**:
1. Transfer Learning with VGG16 (pretrained on ImageNet)
2. Heavy data augmentation
3. Fine-tuning strategy
4. Learning rate scheduling
5. Expected accuracy: 65-75%

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Load and Prepare Data

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/fer2013/fer2013.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

In [ ]:
def prepare_data_for_vgg(df):
    """
    Prepare data for VGG16 (requires 3 channels, not 1)
    VGG16 expects RGB images, so we'll replicate grayscale to 3 channels
    """
    pixels = df['pixels'].tolist()
    
    X = []
    for pixel_sequence in pixels:
        # Convert to 48x48 grayscale
        face = np.array([int(pixel) for pixel in pixel_sequence.split()]).reshape(48, 48)
        
        # Replicate to 3 channels for VGG16 (RGB format)
        face_rgb = np.stack([face, face, face], axis=-1)
        
        X.append(face_rgb)
    
    X = np.array(X).astype('float32') / 255.0  # Normalize
    y = df['emotion'].values
    
    return X, y

print("Preparing data...")

# Split by Usage
train_data = df[df['Usage'] == 'Training']
test_data = df[df['Usage'] == 'PublicTest']

X_train, y_train = prepare_data_for_vgg(train_data)
X_test, y_test = prepare_data_for_vgg(test_data)

# Create validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42,
    stratify=y_train
)

print(f"\nData shapes:")
print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

## 3. Data Augmentation

**Key technique**: Artificially expand training data with transformations

In [ ]:
# Create data augmentation generator for training
train_datagen = ImageDataGenerator(
    rotation_range=20,           # Rotate images randomly by 20 degrees
    width_shift_range=0.1,       # Shift horizontally by 10%
    height_shift_range=0.1,      # Shift vertically by 10%
    horizontal_flip=True,        # Flip images horizontally
    zoom_range=0.1,              # Zoom in/out by 10%
    brightness_range=[0.8, 1.2], # Adjust brightness
    fill_mode='nearest'          # Fill missing pixels after transformations
)

# No augmentation for validation (use original images)
val_datagen = ImageDataGenerator()

# Create generators
batch_size = 64

train_generator = train_datagen.flow(
    X_train, y_train,
    batch_size=batch_size,
    shuffle=True
)

val_generator = val_datagen.flow(
    X_val, y_val,
    batch_size=batch_size,
    shuffle=False
)

print("Data augmentation configured!")
print(f"Training batches per epoch: {len(train_generator)}")
print(f"Validation batches per epoch: {len(val_generator)}")

## 4. Visualize Augmented Data

In [ ]:
# Show augmentation examples
sample_image = X_train[0:1]

plt.figure(figsize=(15, 3))
for i in range(8):
    # Generate augmented version
    augmented = train_datagen.flow(sample_image, batch_size=1)
    aug_image = next(augmented)[0]
    
    plt.subplot(1, 8, i+1)
    plt.imshow(aug_image[:,:,0], cmap='gray')
    plt.axis('off')
    plt.title(f'Aug {i+1}')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Build Transfer Learning Model (VGG16)

**Strategy**: Use VGG16 pretrained on ImageNet as feature extractor

In [ ]:
def create_transfer_learning_model():
    """
    Create model using VGG16 as base
    """
    # Load VGG16 without top layers (pretrained on ImageNet)
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(48, 48, 3)
    )
    
    # Freeze base model layers initially
    base_model.trainable = False
    
    # Add custom classification head
    inputs = keras.Input(shape=(48, 48, 3))
    
    # VGG16 feature extraction
    x = base_model(inputs, training=False)
    
    # Custom layers on top
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(7, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='VGG16_FER2013')
    
    return model

model = create_transfer_learning_model()
model.summary()

## 6. Compile Model

In [ ]:
# Compile with Adam optimizer
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled!")

## 7. Setup Callbacks

In [ ]:
callbacks = [
    # Stop if no improvement
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Save best model
    ModelCheckpoint(
        'best_vgg16_fer2013.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configured!")

## 8. Train Model (Phase 1: Frozen Base)

In [ ]:
print("="*60)
print("PHASE 1: Training with frozen VGG16 base")
print("="*60)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

print("\nPhase 1 training complete!")

## 9. Fine-Tuning (Phase 2: Unfreeze Some Layers)

**Strategy**: Unfreeze top layers of VGG16 for fine-tuning

In [ ]:
print("="*60)
print("PHASE 2: Fine-tuning - unfreezing top layers")
print("="*60)

# Unfreeze the last 4 layers of VGG16
base_model = model.layers[1]  # VGG16 is the second layer
base_model.trainable = True

# Freeze all layers except the last 4
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=0.0001),  # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable layers: {sum([1 for layer in model.layers if layer.trainable])}")
print("Model recompiled with lower learning rate!")

In [ ]:
# Continue training with fine-tuning
history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

print("\nPhase 2 (fine-tuning) complete!")

## 10. Evaluate Model

In [ ]:
print("="*60)
print("FINAL EVALUATION")
print("="*60)

# Evaluate on validation set
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"\nValidation Accuracy: {val_acc*100:.2f}%")
print(f"Test Accuracy:       {test_acc*100:.2f}%")
print(f"\nValidation Loss:     {val_loss:.4f}")
print(f"Test Loss:           {test_loss:.4f}")
print("="*60)

## 11. Visualize Training History

In [ ]:
# Combine both phases
def plot_combined_history(hist1, hist2):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Combine histories
    train_acc = hist1.history['accuracy'] + hist2.history['accuracy']
    val_acc = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
    train_loss = hist1.history['loss'] + hist2.history['loss']
    val_loss = hist1.history['val_loss'] + hist2.history['val_loss']
    
    epochs = range(1, len(train_acc) + 1)
    phase1_end = len(hist1.history['accuracy'])
    
    # Accuracy plot
    ax1.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
    ax1.plot(epochs, val_acc, 'r-', label='Val Accuracy', linewidth=2)
    ax1.axvline(x=phase1_end, color='green', linestyle='--', label='Fine-tuning starts')
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss plot
    ax2.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
    ax2.plot(epochs, val_loss, 'r-', label='Val Loss', linewidth=2)
    ax2.axvline(x=phase1_end, color='green', linestyle='--', label='Fine-tuning starts')
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_combined_history(history_phase1, history_phase2)

## 12. Confusion Matrix

In [ ]:
# Get predictions
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotions, yticklabels=emotions)
plt.title('Confusion Matrix - VGG16 Transfer Learning', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 13. Classification Report

In [ ]:
print("="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, target_names=emotions))
print("="*70)

## 14. Compare: Simple CNN vs Transfer Learning

In [ ]:
print("="*70)
print("APPROACH COMPARISON")
print("="*70)

comparison_data = {
    'Approach': ['Simple CNN (3 conv blocks)', 'VGG16 Transfer Learning'],
    'Parameters': ['~500K', '~15M'],
    'Training Strategy': ['From scratch', '2-phase: freeze + fine-tune'],
    'Data Augmentation': ['None', 'Heavy (rotation, flip, zoom, etc.)'],
    'Expected Accuracy': ['55-65%', '65-75%'],
    'Training Time': ['15-20 min', '30-45 min']
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("\n" + "="*70)

print(f"\nYour VGG16 Model Test Accuracy: {test_acc*100:.2f}%")
print("\nKey Improvements from Transfer Learning:")
print("1. Leverages ImageNet knowledge (pretrained features)")
print("2. Data augmentation creates more training examples")
print("3. Two-phase training prevents overfitting")
print("4. Better generalization to test data")

## 15. Save Model

In [ ]:
model.save('fer2013_vgg16_final.h5')
print("Final model saved as 'fer2013_vgg16_final.h5'")

print("\nTo load:")
print("model = keras.models.load_model('fer2013_vgg16_final.h5')")

## 16. Summary

In [ ]:
print("="*70)
print("SUMMARY - TRANSFER LEARNING APPROACH")
print("="*70)
print("\nTechniques Used:")
print("1. Transfer Learning (VGG16 pretrained on ImageNet)")
print("2. Data Augmentation (rotation, shift, flip, zoom, brightness)")
print("3. Two-Phase Training (freeze then fine-tune)")
print("4. Learning Rate Scheduling")
print("5. Early Stopping & Model Checkpointing")

print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")
print(f"Improvement over simple CNN: ~10-15% typically")

print("\nWhy This Works Better:")
print("- VGG16 learned general visual features from 1M+ images")
print("- Data augmentation prevents overfitting on small dataset")
print("- Fine-tuning adapts pretrained features to emotions")
print("- Two-phase training balances speed and accuracy")
print("="*70)